# Qwen3.8-27B → DeepSeek Harness

**Active model path:** `Qwen/Qwen3.8-27B-FP8`. The Flash-Next experiment is not used by this notebook.

This notebook uses the outbound-only Supabase relay. No reverse tunnel or public Colab port is used. Your existing Windows Harness model id remains `qwen3.8-27b` and the relay id remains `qwen3-8-27b`.

In Colab **Secrets** (key icon), add `QWEN_RELAY_SECRET` and enable **Notebook access**.

For automatic Oracle VM wake-up, also add `ORACLE_WAKE_GITHUB_TOKEN` and enable **Notebook access**.

### Cheapest validation strategy
Before paying for a high-memory GPU, use **Section 2A — Expected-OOM smoke test** on the cheapest compatible **SM80+** GPU you can get. **L4 24 GB is recommended.** It deliberately runs the real 27B FP8 + 262K + MTP3 production stack until VRAM/KV allocation fails. Reaching the expected memory limit is a **PASS**.

Use **Section 2B** for the real high-memory GPU worker. A100 80 GB is the proven target; a larger compatible GPU such as the Colab G4 can also pass the >=70 GB production guard. Do not run Sections 2A, 2B and 3 at the same time.


## Section 1 — Install/update worker
Run this once in a fresh Colab runtime. This installs the current `main` Qwen3.8-27B worker and includes retry/backoff for transient Supabase/Cloudflare gateway errors such as HTTP 520.


In [ ]:
%pip install -q --upgrade "git+https://github.com/Logan17de/All-testing.git#subdirectory=llm"
import importlib.metadata as metadata
print(f"all-testing-llm {metadata.version('all-testing-llm')}: OK ✅")


## Section 2A — CHEAP expected-OOM production-path smoke test
Use this **before the real high-memory worker**. Recommended runtime: **L4 24 GB** or another undersized SM80+ GPU. This runs the **same official Qwen3.8-27B-FP8 model, 262,144 context, FP8 KV, MTP3, FlashInfer, Marlin, CUDA graphs/torch.compile, Supabase preflight and Oracle lease** as production. The only relaxed rule is the 80 GB-class VRAM guard.

Expected result: vLLM reaches real model/KV initialization and fails because the GPU is too small. The cell converts that expected memory failure into `EXPECTED VRAM LIMIT REACHED ✅ — SMOKE TEST PASSED`. If you get any other error first, that is a real code/dependency/configuration problem to fix.


In [ ]:
import qwen3_8_27b_supabase_colab_oom_smoke as qwen_oom_smoke
qwen_oom_smoke.main()


## Optional — capture BEFORE benchmark
Use this only when doing a comparison and an older compatible vLLM server is still running in the same runtime. It measures localhost TTFT and decode tokens/sec and saves the result under `/content/qwen_benchmark_results.json`.


In [ ]:
import qwen3_8_27b_colab_runtime as qwen_fast
qwen_fast.benchmark_running_server("before")


## Section 2B — REAL Qwen3.8-27B optimized worker
This uses official `Qwen/Qwen3.8-27B-FP8` weights, native 262,144 total context, FP8 KV cache, native MTP with 3 draft tokens, prefix caching, chunked prefill, torch.compile/CUDA Graphs, and single-user scheduling. The Colab-safe bootstrap installs and repairs the CUDA-13 vLLM nightly environment before Supabase/relay imports are used, then verifies Torch, Torchvision, Pillow, Transformers, vLLM, FlashInfer and the Qwen MTP modules in a fresh Python process.


In [ ]:
import qwen3_8_27b_colab_runtime as qwen_worker
qwen_worker.main()


## Section 3 — TESTING: API/relay only (NO GPU)
Use a normal CPU Colab runtime. Run **Section 1**, skip Sections 2A and 2B, then run this section. No Torch, vLLM, CUDA, Hugging Face model, or GPU is used. Every request received from Harness is decoded through the real relay and returned as the OpenAI-compatible assistant response **`succeed`**. Leave this cell running while testing Harness.


In [ ]:
import qwen_supabase_test_worker as relay_test
relay_test.main()
